# Notebook 01 — Images and Tensors

Welcome! This is the very first notebook of the course. Before we touch a neural
network we need to be comfortable with the **raw material of computer vision**:
images as numbers, and the PyTorch tensor that carries those numbers around.
By the end of this notebook you will read an image from disk, understand its
layout, normalize it the way pretrained models expect, and display it again.

## Learning goals
- Understand that a digital image is just a grid of numbers with a channel axis.
- Load images with three different libraries (PIL, OpenCV, `torchvision.io`) and
  know why each returns a slightly different object.
- Convert between the two common layouts: `(H, W, C)` (numpy/OpenCV) and
  `(C, H, W)` (PyTorch).
- Normalize an image with ImageNet statistics and be able to **reverse** the
  operation for visualization.
- Browse a small dataset (CIFAR-10) as a preview of what Notebook 02 will use.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/01_images_and_tensors.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device


## 1. What is a digital image?

A colour image is a 3-D grid of numbers:

- **Height (H)** -- the number of pixel rows.
- **Width (W)**  -- the number of pixel columns.
- **Channels (C)** -- the colour planes. For a standard RGB photograph `C = 3`:
  one plane for red, one for green, one for blue. Grayscale images have `C = 1`.

Each value inside the grid is a **pixel intensity**. Stored on disk those are
usually 8-bit unsigned integers (`uint8`, range 0..255). Once we load an image
into a framework like PyTorch we almost always convert it to 32-bit floats in
`[0, 1]` or a normalized range — that is what makes gradient-based learning
well-behaved.

Let's download the CIFAR-10 dataset (it's tiny, ~170 MB, and used by many of our
later notebooks) and pick one image from it to explore.


In [ ]:
# CIFAR-10 contains 60,000 32x32 colour images across 10 classes.
# We download it via torchvision; the data lands in ../data (shared across
# notebooks in this repo).
import os
from utils.env import data_dir
from torchvision.datasets import CIFAR10

DATA_DIR = data_dir()
print("Data directory:", DATA_DIR)

train_ds = CIFAR10(root=DATA_DIR, train=True, download=True)
print("Number of training images:", len(train_ds))
print("Class names:", train_ds.classes)

# A single item is a (PIL.Image, label) tuple.
img_pil, label = train_ds[0]
print("First item class:", train_ds.classes[label])
print("PIL image type:  ", type(img_pil).__name__)
print("PIL image size:  ", img_pil.size, "  (width, height) -- note the order!")
print("PIL image mode:  ", img_pil.mode, "  -- 'RGB' means 3 channels, 8 bits each")


### Peek at the raw pixel values

Before we show the image, let's **look at the numbers**. If you've never
programmed with images before, the single most important thing to internalize is
that everything below is ultimately a big NumPy array.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convert the PIL image to a numpy array. Layout: (H, W, C) with uint8 values.
img_np = np.array(img_pil)
print("img_np.shape =", img_np.shape)      # (32, 32, 3)
print("img_np.dtype =", img_np.dtype)      # uint8
print("min, max     =", img_np.min(), img_np.max())

# First 5x5x3 slice -- top-left corner, all three channels.
# Each inner triple is (R, G, B) for one pixel.
print("Top-left 5x5 RGB values:")
print(img_np[:5, :5, :])

# And show the image itself (matplotlib expects H,W,C in [0,1] or uint8).
plt.figure(figsize=(3, 3))
plt.imshow(img_np)
plt.title(f"label = {train_ds.classes[label]}")
plt.axis("off");


## 2. Loading images three ways

In practice you'll meet three popular ways to load an image:

| Library            | Returns              | Channel order | Dtype  | Notes |
|--------------------|----------------------|---------------|--------|-------|
| `PIL.Image.open`   | `PIL.Image`          | RGB           | uint8  | Lazy loading; you still need `.convert("RGB")` sometimes. |
| `cv2.imread`       | `numpy.ndarray`      | **BGR**       | uint8  | Very fast, but the BGR channel order trips up *everyone*. |
| `torchvision.io.read_image` | `torch.Tensor` | RGB           | uint8  | Already in `(C, H, W)` layout -- PyTorch's native shape. |

To compare them we need an image file on disk (CIFAR-10 images live inside a
pickle, so let's save our one sample as a PNG first).


In [ ]:
import os
sample_path = os.path.join(DATA_DIR, "cifar_sample.png")
img_pil.save(sample_path)
print("Saved sample image ->", sample_path)


In [ ]:
# --- Method 1: PIL -----------------------------------------------------------
from PIL import Image
pil_img = Image.open(sample_path).convert("RGB")
pil_arr = np.array(pil_img)                    # H,W,C uint8
print("PIL shape:", pil_arr.shape, "dtype:", pil_arr.dtype)
print("PIL first pixel (R,G,B):", pil_arr[0, 0])

# --- Method 2: OpenCV --------------------------------------------------------
# cv2 imports lazily; if you don't have it locally, `pip install opencv-python`.
import cv2
bgr = cv2.imread(sample_path)                  # H,W,C uint8 in BGR order
print("cv2 shape:", bgr.shape, "dtype:", bgr.dtype)
print("cv2 first pixel (B,G,R):", bgr[0, 0])

# OpenCV's BGR -> RGB fix:
rgb_from_cv = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
print("cv2->RGB first pixel:", rgb_from_cv[0, 0])

# --- Method 3: torchvision.io.read_image ------------------------------------
from torchvision.io import read_image
tv_tensor = read_image(sample_path)            # C,H,W uint8 tensor
print("tv shape:", tuple(tv_tensor.shape), "dtype:", tv_tensor.dtype)
print("tv first pixel (R,G,B):", tv_tensor[:, 0, 0].tolist())


All three libraries agree on the pixel *values* (give or take the BGR flip) but
they disagree on the **container type** and the **axis order**. Mixing them up
is a classic source of bugs: if you display a cv2 image without converting
BGR -> RGB, the colours will look off; if you feed an `(H, W, C)` numpy array to
a PyTorch conv layer it will complain.


## 3. Tensors: the PyTorch shape convention

A PyTorch tensor is basically a NumPy array that can live on a GPU and remember
how it was computed (so autograd can back-propagate through it). For our
purposes right now, think of it as a typed multi-dimensional array.

### The shape rule you will use every day

- **NumPy / OpenCV / PIL**: `(H, W, C)` -- channel is the **last** axis.
- **PyTorch**: `(C, H, W)` -- channel is the **first** axis.
  - For a batch: `(N, C, H, W)` where N is the batch size.

Why the difference? PyTorch uses this layout because the underlying CUDA /
cuDNN primitives are written to operate channel-first. There is no deep
mathematical reason -- it's just a convention you must obey.


In [ ]:
import torch

# Start with a H,W,C numpy array (from PIL).
print("numpy HWC shape:", pil_arr.shape)

# Convert to a tensor and permute axes to C,H,W.
hwc = torch.from_numpy(pil_arr)          # still (H, W, C)
chw = hwc.permute(2, 0, 1)               # (C, H, W)
print("tensor HWC shape:", tuple(hwc.shape))
print("tensor CHW shape:", tuple(chw.shape))

# Add a batch dimension at the front -- this is what a DataLoader would produce.
batched = chw.unsqueeze(0)               # (1, C, H, W)
print("batched shape:", tuple(batched.shape))

# Back to float in [0, 1]: divide by 255.
float_img = chw.float() / 255.0
print("float dtype:", float_img.dtype, "min:", float_img.min().item(), "max:", float_img.max().item())


## 4. Normalization -- why, and how

After converting pixels to `[0, 1]` floats, models almost always apply a second
step: **normalization** with a per-channel mean and standard deviation.

```
x_norm[c] = (x[c] - mean[c]) / std[c]
```

### Why?

1. **Better optimization**: inputs centered around zero with unit variance
   make gradient descent less sensitive to the learning rate. All else equal,
   you'll converge faster and more reliably.
2. **Pretrained compatibility**: every ImageNet-pretrained model (including
   EfficientNetV2) was trained on inputs normalized with
   `mean = [0.485, 0.456, 0.406]`, `std = [0.229, 0.224, 0.225]`.
   If you feed it un-normalized data at inference, it will still produce
   *some* answer, but the accuracy drops significantly.

Let's apply this normalization and inspect the effect visually.


In [ ]:
# ImageNet per-channel statistics.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

normalized = (float_img - IMAGENET_MEAN) / IMAGENET_STD

# Show summary stats per channel, before and after.
print("Before normalization (per channel):")
for c, name in enumerate(["R", "G", "B"]):
    print(f"  {name}: mean={float_img[c].mean().item():.3f}  std={float_img[c].std().item():.3f}")
print("After normalization (per channel):")
for c, name in enumerate(["R", "G", "B"]):
    print(f"  {name}: mean={normalized[c].mean().item():.3f}  std={normalized[c].std().item():.3f}")


In [ ]:
# Histograms of pixel values before and after normalization.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(float_img.flatten().numpy(), bins=40, color="steelblue")
axes[0].set_title("Before: pixels in [0, 1]")
axes[0].set_xlabel("value"); axes[0].set_ylabel("count")

axes[1].hist(normalized.flatten().numpy(), bins=40, color="seagreen")
axes[1].set_title("After: ~zero mean, ~unit std")
axes[1].set_xlabel("value"); axes[1].set_ylabel("count")
fig.tight_layout();


## 5. Visualizing tensors (unnormalize first!)

Once an image has been normalized, `plt.imshow` of its raw values looks bad:
the tensor contains negative numbers and values > 1, which matplotlib will
clip/stretch in confusing ways.

The fix is to *undo* the normalization before display:

```
x = x_norm * std + mean
```

This is such a common step that we'll write a small helper for it.


In [ ]:
def denorm(t: torch.Tensor, mean, std) -> torch.Tensor:
    """Reverse Normalize(mean, std). Works for (C,H,W) or (N,C,H,W) tensors."""
    m = torch.as_tensor(mean).view(-1, 1, 1)
    s = torch.as_tensor(std).view(-1, 1, 1)
    return t * s + m

# Show raw-normalized tensor vs denormalized side-by-side.
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

# If we display the normalized tensor directly, we have to permute back to HWC.
bad_view = normalized.permute(1, 2, 0).numpy()
axes[0].imshow(bad_view)
axes[0].set_title("Normalized (looks wrong)")
axes[0].axis("off")

fixed = denorm(normalized, IMAGENET_MEAN.flatten(), IMAGENET_STD.flatten())
good_view = fixed.permute(1, 2, 0).clamp(0, 1).numpy()
axes[1].imshow(good_view)
axes[1].set_title("Denormalized")
axes[1].axis("off")
fig.tight_layout();


## 6. Dataset preview -- a 4x4 grid of CIFAR-10

As a preview of Notebook 02, let's grab 16 random CIFAR-10 images and show them
in a 4x4 grid. We won't train anything yet -- this is just to reinforce the idea
that a dataset is, conceptually, a list of `(image, label)` pairs.


In [ ]:
import random

indices = random.sample(range(len(train_ds)), 16)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, idx in zip(axes.flat, indices):
    img, lbl = train_ds[idx]
    ax.imshow(np.array(img))
    ax.set_title(train_ds.classes[lbl], fontsize=9)
    ax.axis("off")
fig.suptitle("CIFAR-10 preview (16 random training images)", y=1.02)
fig.tight_layout();


## Key Takeaways

- An image is a 3-D grid of numbers: height, width, channels. Grayscale has
  `C = 1`, RGB has `C = 3`.
- Numpy/PIL/OpenCV use `(H, W, C)`; PyTorch uses `(C, H, W)` -- convert with
  `.permute(2, 0, 1)`.
- OpenCV returns BGR; remember to swap to RGB.
- Normalization subtracts a per-channel mean and divides by a per-channel std.
  Use ImageNet stats when you plan to fine-tune an ImageNet-pretrained model.
- To *display* a normalized tensor you must first **denormalize** it, then
  permute back to `(H, W, C)`.


## Exercises

1. **Write your own `normalize` function** that accepts a `(C, H, W)` tensor and
   a `(mean, std)` pair and returns the normalized tensor. Verify that chaining
   it with the `denorm` helper in this notebook recovers the original image to
   within `1e-6`.
2. **Load a local image of your own** using all three libraries (PIL, OpenCV,
   `torchvision.io`). Confirm the shapes, dtypes and channel orders match the
   table in Section 2.
3. **Compute channel statistics** (mean and std per channel) over the first
   5,000 CIFAR-10 training images. Compare your values to the ImageNet stats
   we used above -- are they similar?
